# STEP 1: Overview — Script Description
# ------------------------------------
# This script processes all building folders inside the "baseline_models" directory.
# For each building:
#   1. It locates the "TOTAL/annual_savings.csv" file.
#   2. Reads the ENERGY, CARBON, COST_FIXED, COST_FLEX values.
#   3. Computes savings columns using the specified dependency chain:
#        BASELINE → WINDOWS/WALLS/ROOF/FAB → HP → SOLAR_THERMAL → PV → BATTERY
#   4. Writes the computed results back into the same CSV file.
#
# The script is robust to missing files and logs progress for troubleshooting.

In [1]:
# STEP 2: Import Libraries and Define Paths
# ----------------------------------------

import os
import pandas as pd

# Root directory containing all building folders
BASE_DIR = "/Users/jakegehrung/Desktop/data_raw/baseline_models"

In [2]:
# STEP 3: Define Calculation Logic
# --------------------------------

def compute_savings(df):
    """
    Compute savings columns based on dependency chain.
    """

    # Ensure LABEL is index for easier lookup
    df = df.set_index("LABEL")

    # Extract baseline values
    a, b, c, d = df.loc["BASELINE", ["ENERGY", "CARBON", "COST_FIXED", "COST_FLEX"]]

    # Initialize savings columns
    df["ENERGY_SAVINGS"] = None
    df["CARBON_SAVINGS"] = None
    df["COST_SAVINGS_FIXED"] = None
    df["COST_SAVINGS_FLEX"] = None

    # Helper function
    def calc(prev_vals, current_label):
        curr = df.loc[current_label, ["ENERGY", "CARBON", "COST_FIXED", "COST_FLEX"]]
        return prev_vals - curr

    # WINDOWS, WALLS, ROOF, FAB (relative to baseline)
    for label in ["WINDOWS", "WALLS", "ROOF", "FAB"]:
        if label in df.index:
            diff = calc(pd.Series([a, b, c, d], index=["ENERGY", "CARBON", "COST_FIXED", "COST_FLEX"]), label)
            df.loc[label, ["ENERGY_SAVINGS", "CARBON_SAVINGS", "COST_SAVINGS_FIXED", "COST_SAVINGS_FLEX"]] = diff.values

    # FAB base for next step
    if "FAB" in df.index:
        prev = df.loc["FAB", ["ENERGY", "CARBON", "COST_FIXED", "COST_FLEX"]]
    else:
        prev = pd.Series([a, b, c, d], index=["ENERGY", "CARBON", "COST_FIXED", "COST_FLEX"])

    # HP
    if "HP" in df.index:
        diff = prev - df.loc["HP", ["ENERGY", "CARBON", "COST_FIXED", "COST_FLEX"]]
        df.loc["HP", ["ENERGY_SAVINGS", "CARBON_SAVINGS", "COST_SAVINGS_FIXED", "COST_SAVINGS_FLEX"]] = diff.values
        prev = df.loc["HP", ["ENERGY", "CARBON", "COST_FIXED", "COST_FLEX"]]

    # SOLAR_THERMAL
    if "SOLAR_THERMAL" in df.index:
        diff = prev - df.loc["SOLAR_THERMAL", ["ENERGY", "CARBON", "COST_FIXED", "COST_FLEX"]]
        df.loc["SOLAR_THERMAL", ["ENERGY_SAVINGS", "CARBON_SAVINGS", "COST_SAVINGS_FIXED", "COST_SAVINGS_FLEX"]] = diff.values
        prev = df.loc["SOLAR_THERMAL", ["ENERGY", "CARBON", "COST_FIXED", "COST_FLEX"]]

    # PV
    if "PV" in df.index:
        diff = prev - df.loc["PV", ["ENERGY", "CARBON", "COST_FIXED", "COST_FLEX"]]
        df.loc["PV", ["ENERGY_SAVINGS", "CARBON_SAVINGS", "COST_SAVINGS_FIXED", "COST_SAVINGS_FLEX"]] = diff.values
        prev = df.loc["PV", ["ENERGY", "CARBON", "COST_FIXED", "COST_FLEX"]]

    # BATTERY
    if "BATTERY" in df.index:
        diff = prev - df.loc["BATTERY", ["ENERGY", "CARBON", "COST_FIXED", "COST_FLEX"]]
        df.loc["BATTERY", ["ENERGY_SAVINGS", "CARBON_SAVINGS", "COST_SAVINGS_FIXED", "COST_SAVINGS_FLEX"]] = diff.values

    # Reset index for saving
    return df.reset_index()

In [3]:
# STEP 4: Process All Buildings
# -----------------------------

processed = 0
skipped = 0

for building_id in os.listdir(BASE_DIR):
    total_path = os.path.join(BASE_DIR, building_id, "TOTAL", "annual_savings.csv")

    if not os.path.exists(total_path):
        print(f"Skipping {building_id} (file not found)")
        skipped += 1
        continue

    try:
        df = pd.read_csv(total_path)

        # Compute savings
        df_updated = compute_savings(df)

        # Save back to file
        df_updated.to_csv(total_path, index=False)

        print(f"Processed: {building_id}")
        processed += 1

    except Exception as e:
        print(f"Error in {building_id}: {e}")
        skipped += 1

print("\n--- SUMMARY ---")
print(f"Processed: {processed}")
print(f"Skipped: {skipped}")

Processed: 1001664924
Processed: 1001829085
Processed: 1001063639
Processed: 1001829071
Processed: 1234761002
Processed: 1002539407
Processed: 1001063637
Processed: 1001664941
Processed: 1001991633
Processed: 1235057414
Processed: 1001829079
Processed: 1001664922
Processed: 1234761003
Processed: 1001664925
Processed: 1000238907
Processed: 1234761004
Processed: 1002313096
Processed: 1001870878
Processed: 1001664940
Processed: 1234982990
Processed: 1234806523
Processed: 1001870876
Processed: 1001870882
Processed: 1002143357
Processed: 1001951902
Processed: 1234621926
Skipping .DS_Store (file not found)
Processed: 1234647955
Processed: 1001906269
Processed: 1001951903
Processed: 1001627539
Processed: 1002143351
Processed: 1001627541
Processed: 1236594950
Processed: 1001627570
Processed: 1002031280
Processed: 1001627549
Processed: 1001627540
Processed: 1001627547
Processed: 1001870854
Processed: 1001430744
Processed: 1234760995
Processed: 1235812262
Processed: 1000336709
Processed: 1001991

In [4]:
# STEP 5: (Optional) Validate One File
# ------------------------------------
# Use this to manually inspect a specific building after processing

test_path = "/Users/jakegehrung/Desktop/data_raw/baseline_models/1001829061/TOTAL/annual_savings.csv"

df_test = pd.read_csv(test_path)
df_test

,LABEL,ENERGY,CARBON,COST_FIXED,COST_FLEX,ENERGY_SAVINGS,CARBON_SAVINGS,COST_SAVINGS_FIXED,COST_SAVINGS_FLEX
0,BASELINE,3.585215e+07,4805.286772,2798.054673,2645.146898,NaN,NaN,NaN,NaN
1,WINDOWS,3.831054e+07,5147.494486,2919.110912,2766.203137,-2.458390e+06,-342.207714,-121.056239,-121.056239
2,WALLS,3.794446e+07,5093.087983,2901.028718,2748.120943,-2.092308e+06,-287.801212,-102.974045,-102.974045
3,ROOF,4.035413e+07,5428.592707,3019.687208,2866.779433,-4.501979e+06,-623.305935,-221.632535,-221.632535
4,FAB,3.585215e+07,4805.286772,2798.054673,2645.146898,0.000000e+00,0.000000,0.000000,0.000000
5,HP,1.809390e+07,1882.931413,2604.935597,2274.122079,1.775825e+07,2922.355358,193.119076,371.024819
6,SOLAR_THERMAL,1.487179e+07,1402.836645,2445.763237,2114.949719,3.222113e+06,480.094769,159.172360,159.172360
7,PV,1.487179e+07,1402.836645,2445.763237,2114.949719,0.000000e+00,0.000000,0.000000,0.000000
8,BATTERY,1.748337e+07,1580.424132,2262.779286,1919.830617,-2.611581e+06,-177.587487,182.983951,195.119103
